In [1]:
# Cai FAISS de tao index tim kiem vector va cai OpenAI CLIP de load model CLIP.
!pip install faiss-cpu -q
!pip install git+https://github.com/openai/CLIP.git -q

import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import clip
import faiss

# Tu dong dung GPU neu Kaggle session co bat accelerator.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-c

In [2]:
PROCESSED_DIR = Path("/kaggle/input/datasets/lehuuluongb2306557/fashion200k-preprocessed/processed")
IMAGE_ROOT = PROCESSED_DIR / "images"
CHECKPOINT_DIR = Path("/kaggle/input/notebooks/lehuuluongb2306557/fashion-dataset/checkpoints")

# Tat ca file index/embedding se duoc ghi vao /kaggle/working de co the download sau khi chay xong.
OUTPUT_DIR = Path("/kaggle/working/index_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLIP_MODEL = "ViT-B/32"
BATCH_SIZE = 128
NUM_WORKERS = 2

# Dung cho embedding cua item trong database:
# item_vector = ALPHA * image_embedding + (1 - ALPHA) * text_embedding
ALPHA = 0.6

# "all" de build index cho app cuoi cung.
# Co the doi thanh "train" neu chi muon danh gia nghiem ngat tren train items.
INDEX_SPLIT = "all"

print(f"PROCESSED_DIR exists? {PROCESSED_DIR.exists()}")
print(f"IMAGE_ROOT exists?    {IMAGE_ROOT.exists()}")
print(f"CHECKPOINT_DIR exists? {CHECKPOINT_DIR.exists()}")

PROCESSED_DIR exists? True
IMAGE_ROOT exists?    True
CHECKPOINT_DIR exists? True


In [3]:
# Khai bao dung 3 file da tao o buoc preprocess.
meta_path = PROCESSED_DIR / "metadata.csv"
interactions_path = PROCESSED_DIR / "interactions.csv"
splits_path = PROCESSED_DIR / "splits.json"

# Dung assert de bao loi som neu Kaggle chua add dung dataset.
assert meta_path.exists(), f"Khong tim thay {meta_path}"
assert interactions_path.exists(), f"Khong tim thay {interactions_path}"
assert splits_path.exists(), f"Khong tim thay {splits_path}"

df_meta = pd.read_csv(meta_path)
df_interactions = pd.read_csv(interactions_path)

with open(splits_path, "r", encoding="utf-8") as f:
    splits = json.load(f)

print(f"metadata.csv: {len(df_meta):,} san pham")
print(f"interactions.csv: {len(df_interactions):,} tuong tac")
print({k: len(v) for k, v in splits.items()})

# Kiem tra schema giup tranh loi sai ten cot khi encode.
required_meta_cols = {
    "item_id", "product_id", "name", "description",
    "category", "subcategory", "gender", "image_path"
}
missing_meta_cols = required_meta_cols - set(df_meta.columns)
assert not missing_meta_cols, f"metadata.csv thieu cot: {missing_meta_cols}"

required_inter_cols = {"user_id", "item_id", "rating"}
missing_inter_cols = required_inter_cols - set(df_interactions.columns)
assert not missing_inter_cols, f"interactions.csv thieu cot: {missing_inter_cols}"

# item_id phai la khoa duy nhat de map vi tri FAISS -> san pham that
assert df_meta["item_id"].is_unique, "item_id trong metadata.csv bi trung"
assert set(df_interactions["item_id"]).issubset(set(df_meta["item_id"])), "interactions co item_id khong nam trong metadata"

split_item_ids = set()
for split_name, ids in splits.items():
    overlap = split_item_ids.intersection(ids)
    assert not overlap, f"Split {split_name} bi trung item voi split khac"
    split_item_ids.update(ids)

# Dam bao train/val/test phu het metadata va khong bi thieu item nao.
assert split_item_ids == set(df_meta["item_id"]), "splits.json khong khop day du voi metadata.csv"

metadata.csv: 65,879 san pham
interactions.csv: 29,785 tuong tac
{'train': 52703, 'val': 6588, 'test': 6588}


In [4]:
# Chuan hoa description truoc khi tokenize bang CLIP.
df_meta["description"] = df_meta["description"].fillna("").astype(str).str.strip()

def remap_image_path(raw_path) -> str:
    # Neu path trong CSV da ton tai truc tiep thi dung luon.
    raw = Path(str(raw_path))
    if raw.exists():
        return str(raw)

    candidates = [IMAGE_ROOT / raw.name]

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    return str(candidates[0])

# Tao mapping item_id -> train/val/test de luu lai split trong metadata_indexed.csv.
item_to_split = {}
for split_name, ids in splits.items():
    for item_id in ids:
        item_to_split[int(item_id)] = split_name

df_meta["split"] = df_meta["item_id"].map(item_to_split)
df_meta["image_path"] = df_meta["image_path"].apply(remap_image_path)

# Chi encode nhung item co anh doc duoc, co mo ta va co split hop le.
image_exists = df_meta["image_path"].map(lambda p: Path(p).is_file())
has_caption = df_meta["description"].ne("")
valid_mask = image_exists & has_caption & df_meta["split"].notna()

removed = int((~valid_mask).sum())
if removed:
    print(f"Loai {removed} san pham thieu anh, mo ta hoac split")

df_meta = df_meta.loc[valid_mask].reset_index(drop=True)

# Mac dinh build index tren toan bo item de dung cho app.
# Neu can danh gia nghiem ngat, doi INDEX_SPLIT thanh train/val/test.
if INDEX_SPLIT != "all":
    assert INDEX_SPLIT in {"train", "val", "test"}, "INDEX_SPLIT phai la all/train/val/test"
    df_meta = df_meta[df_meta["split"].eq(INDEX_SPLIT)].reset_index(drop=True)

# Luu thu tu item_id dung bang thu tu vector se them vao FAISS.
item_ids = df_meta["item_id"].to_numpy(dtype=np.int64)

print(f"So san pham se encode: {len(df_meta):,}")
print(df_meta["split"].value_counts().to_string())

So san pham se encode: 65,879
split
train    52703
test      6588
val       6588


In [5]:
best_ckpt_path = CHECKPOINT_DIR / "fashion_clip_best.pt"
assert best_ckpt_path.exists(), f"Khong tim thay {best_ckpt_path}"

# Load backbone CLIP goc, sau do nap lai trong so da fine-tune.
clip_model, clip_preprocess = clip.load(CLIP_MODEL, device=DEVICE, jit=False)

# weights_only=False giup doc duoc checkpoint co ca metadata nhu epoch/val_loss.
try:
    ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
except TypeError:
    ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
print("Checkpoint keys:", ckpt.keys() if isinstance(ckpt, dict) else type(ckpt))

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
else:
    # Truong hop checkpoint chi la state_dict thuan.
    state_dict = ckpt

clip_model.load_state_dict(state_dict)
clip_model.eval()

epoch = ckpt.get("epoch", "?") if isinstance(ckpt, dict) else "?"
val_loss = ckpt.get("val_loss", None) if isinstance(ckpt, dict) else None

if val_loss is None:
    print(f"Da load best model, epoch={epoch}")
else:
    print(f"Da load best model, epoch={epoch}, val_loss={val_loss:.4f}")

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 132MiB/s]


Checkpoint keys: dict_keys(['format_version', 'epoch', 'model_name', 'model_state_dict', 'val_loss', 'config'])
Da load best model, epoch=3, val_loss=0.6191


In [6]:
class FullEncodeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocess):
        # reset_index de idx trong Dataset trung voi vi tri vector trong FAISS.
        self.df = df.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = Path(row["image_path"])

        try:
            # CLIP preprocess gom resize, center crop va normalize dung chuan CLIP.
            with Image.open(img_path) as img:
                image_tensor = self.preprocess(img.convert("RGB"))
        except Exception as exc:
            raise RuntimeError(f"Khong the doc anh: {img_path}") from exc

        # truncate=True giup caption dai khong vuot qua gioi han token cua CLIP.
        text_tokens = clip.tokenize([row["description"]], truncate=True).squeeze(0)
        item_id = torch.tensor(int(row["item_id"]), dtype=torch.long)

        return image_tensor, text_tokens, item_id


encode_dataset = FullEncodeDataset(df_meta, clip_preprocess)
encode_loader = DataLoader(
    encode_dataset,
    batch_size=BATCH_SIZE,
    # Khong shuffle vi thu tu vector phai khop voi item_ids va metadata_indexed.csv.
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

print(f"So batch: {len(encode_loader)} (batch_size={BATCH_SIZE})")

So batch: 515 (batch_size=128)


In [ ]:
# Khong tinh gradient de tiet kiem VRAM va tang toc encode.
@torch.no_grad()
def encode_full_dataset(model, loader):
    #model o che do danh gia, khong phai training
    model.eval()
    image_feats = []     #embedding anh
    text_feats = []      #embedding mo ta
    ids_collected = []   #danh sach item theo thu tu encode

    #do thoi gian encode
    t_start = time.time()

    for step, (images, texts, ids) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        texts = texts.to(DEVICE, non_blocking=(DEVICE == "cuda"))

        # Tao embedding rieng cho anh va rieng cho mo ta.
        img_feat = model.encode_image(images)
        txt_feat = model.encode_text(texts)

        # Normalize de inner product trong FAISS tuong duong cosine similarity.
        img_feat = F.normalize(img_feat.float(), dim=-1).cpu()
        txt_feat = F.normalize(txt_feat.float(), dim=-1).cpu()

        image_feats.append(img_feat)
        text_feats.append(txt_feat)
        ids_collected.append(ids)

        if (step + 1) % 50 == 0:
            elapsed = time.time() - t_start
            print(f"Batch {step + 1}/{len(loader)} | {elapsed:.0f}s")

    image_feats = torch.cat(image_feats).numpy().astype("float32")
    text_feats = torch.cat(text_feats).numpy().astype("float32")
    ids_collected = torch.cat(ids_collected).numpy().astype(np.int64)

    elapsed = time.time() - t_start
    print(f"Hoan thanh encode {len(ids_collected):,} san pham trong {elapsed:.0f}s")

    return image_feats, text_feats, ids_collected


image_embeddings, text_embeddings, encoded_item_ids = encode_full_dataset(clip_model, encode_loader)

# Neu assert nay fail thi thu tu DataLoader/index da bi lech, search se tra sai item.
assert np.array_equal(encoded_item_ids, item_ids), "Thu tu item_id khong khop voi df_meta"
print(f"image_embeddings shape: {image_embeddings.shape}")
print(f"text_embeddings shape : {text_embeddings.shape}")

Batch 50/515 | 19s
Batch 100/515 | 36s
Batch 150/515 | 53s
Batch 200/515 | 72s
Batch 250/515 | 90s
Batch 300/515 | 109s
Batch 350/515 | 127s
Batch 400/515 | 145s
Batch 450/515 | 163s
Batch 500/515 | 181s
Hoan thanh encode 65,879 san pham trong 186s
image_embeddings shape: (65879, 512)
text_embeddings shape : (65879, 512)


In [8]:
# Gop image embedding va text embedding thanh mot vector dai dien cho moi san pham.
combined_embeddings = ALPHA * image_embeddings + (1 - ALPHA) * text_embeddings

# Sau khi cong co trong so, vector khong con unit norm nen can normalize lai.
norms = np.linalg.norm(combined_embeddings, axis=1, keepdims=True)
assert np.all(norms > 0), "Co vector hybrid co norm bang 0"

combined_embeddings = (combined_embeddings / norms).astype("float32")

embed_dim = combined_embeddings.shape[1]

# IndexFlatIP la exact search: cham hon IVF/HNSW nhung chinh xac va du nhanh voi ~65K item.
index = faiss.IndexFlatIP(embed_dim)
index.add(combined_embeddings)

print(f"FAISS index: {index.ntotal:,} vectors, {embed_dim} chieu")

FAISS index: 65,879 vectors, 512 chieu


In [9]:
# Luu embedding rieng de co the debug hoac rebuild index ma khong can encode lai.
np.save(OUTPUT_DIR / "image_embeddings.npy", image_embeddings)
np.save(OUTPUT_DIR / "text_embeddings.npy", text_embeddings)
np.save(OUTPUT_DIR / "combined_embeddings.npy", combined_embeddings)
np.save(OUTPUT_DIR / "item_ids.npy", encoded_item_ids)

# metadata_indexed.csv phai giu cung thu tu voi vector trong FAISS.
df_meta.to_csv(OUTPUT_DIR / "metadata_indexed.csv", index=False)
df_interactions.to_csv(OUTPUT_DIR / "interactions.csv", index=False)
faiss.write_index(index, str(OUTPUT_DIR / "fashion.index"))

# File config giup app tuan sau biet index nay duoc tao bang model/checkpoint/alpha nao.
config = {
    "clip_model": CLIP_MODEL,
    "checkpoint_file": str(best_ckpt_path),
    "alpha": ALPHA,
    "index_split": INDEX_SPLIT,
    "embedding_dim": int(embed_dim),
    "num_vectors": int(index.ntotal),
    "faiss_metric": "inner_product_on_l2_normalized_vectors",
    "query_note": "For image-only query, use image embedding. For text-only query, use text embedding. For image+text query, use the same alpha-weighted hybrid formula.",
}

with open(OUTPUT_DIR / "index_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f"Da luu artifacts vao {OUTPUT_DIR}")

Da luu artifacts vao /kaggle/working/index_artifacts


In [10]:
@torch.no_grad()
def encode_text_query(text: str):
    # Query text-only: encode cau nguoi dung nhap bang text encoder cua CLIP.
    tokens = clip.tokenize([text], truncate=True).to(DEVICE)
    feat = clip_model.encode_text(tokens)
    feat = F.normalize(feat.float(), dim=-1)
    return feat.cpu().numpy().astype("float32")


@torch.no_grad()
def encode_image_query(image_path):
    # Query image-only: preprocess anh moi giong het luc encode database.
    with Image.open(image_path) as img:
        image_tensor = clip_preprocess(img.convert("RGB")).unsqueeze(0).to(DEVICE)
    feat = clip_model.encode_image(image_tensor)
    feat = F.normalize(feat.float(), dim=-1)
    return feat.cpu().numpy().astype("float32")


def make_hybrid_query(image_path=None, text=None, alpha=ALPHA):
    # Ho tro 3 kieu search: chi anh, chi text, hoac anh + text.
    if image_path is None and text is None:
        #Khong co anh thi tra ve loi
        raise ValueError("Can co image_path hoac text")

    if image_path is not None and text is not None:
        #Co anh + text
        img_feat = encode_image_query(image_path)
        txt_feat = encode_text_query(text)
        query = alpha * img_feat + (1 - alpha) * txt_feat
        query = query / np.linalg.norm(query, axis=1, keepdims=True)
        return query.astype("float32")

    if image_path is not None:
        #Chi co anh
        return encode_image_query(image_path)

    #Chi co text
    return encode_text_query(text)


def search(query_vec, top_k=10):
    # FAISS tra ve indices la vi tri dong trong df_meta/metadata_indexed.csv.
    scores, indices = index.search(query_vec.astype("float32"), top_k)
    rows = df_meta.iloc[indices[0]].copy()
    rows["score"] = scores[0]
    return rows[["item_id", "product_id", "category", "subcategory", "gender", "description", "image_path", "score"]]

In [11]:
# Test bang cau query moi de gan voi cach app that se search bang text.
query = "black floral print dress"
query_vec = make_hybrid_query(text=query)
results = search(query_vec, top_k=10)

print(f"QUERY: {query}")
print(results[["item_id", "category", "subcategory", "description", "score"]].to_string(index=False))

QUERY: black floral print dress
 item_id category             subcategory                       description    score
   16197  dresses        cocktail_dresses          black floral print dress 0.726163
   29653  dresses  casual_and_day_dresses                black floral dress 0.724245
   21898  dresses        cocktail_dresses          black floral print dress 0.723322
   28374  dresses prom_and_formal_dresses                black floral dress 0.721613
   36553  dresses prom_and_formal_dresses          black floral-print dress 0.721562
   29613  dresses  casual_and_day_dresses                black floral dress 0.718359
   34309  dresses   maxi_and_long_dresses     multicolor floral print dress 0.715594
   24892  dresses   maxi_and_long_dresses     multicolor floral print dress 0.713730
   31421  dresses        cocktail_dresses     multicolor floral print dress 0.713382
   37209  dresses prom_and_formal_dresses black floral printed a line dress 0.705454


In [12]:
def show_item_query_results(query_idx, top_k=10):
    # Test sanity bang vector co san trong index: ket qua dau tien thuong la chinh no.
    query_vec = combined_embeddings[query_idx:query_idx + 1]
    scores, indices = index.search(query_vec, top_k + 1)

    query_row = df_meta.iloc[query_idx]
    print(f"QUERY ITEM: [{query_row['category']}] {query_row['description'][:80]}")
    
    shown = 0
    for idx, score in zip(indices[0], scores[0]):
        #Ket qua la chinh no thi bo qua
        if idx == query_idx:
            continue
        row = df_meta.iloc[idx]
        print(f"{shown + 1:>2}. score={score:.3f} | [{row['category']}] {row['description'][:70]}")
        shown += 1
        if shown >= top_k:
            break
    print("-" * 90)

np.random.seed(42)
sample_indices = np.random.choice(len(df_meta), size=min(5, len(df_meta)), replace=False)

for idx in sample_indices:
    show_item_query_results(int(idx), top_k=10)

QUERY ITEM: [jackets] black fur-trimmed hooded vest
 1. score=0.939 | [jackets] black rabbit fur cinched-waist vest
 2. score=0.937 | [jackets] black faux fur-trimmed vest
 3. score=0.937 | [jackets] black shearling fur-trim leather vest
 4. score=0.936 | [jackets] black chicago faux-fur vest
 5. score=0.932 | [jackets] black shearling fur-trim leather vest
 6. score=0.931 | [jackets] black sleeveless ribbed knit fur vest
 7. score=0.931 | [jackets] black fox fur sleeveless vest
 8. score=0.928 | [jackets] black faux-fur sweater vest
 9. score=0.927 | [jackets] black silver fox fur vest
10. score=0.927 | [jackets] black faux fur open-front vest
------------------------------------------------------------------------------------------
QUERY ITEM: [pants] white frame hi-rise leggings
 1. score=0.875 | [pants] white le high skinny
 2. score=0.874 | [pants] optic white overdye high-rise skinny leg
 3. score=0.873 | [pants] white new project legging
 4. score=0.873 | [pants] white luster tw

In [13]:
#Chon so lan test search
n_trials = min(100, len(combined_embeddings))

# Warm-up
_ = index.search(combined_embeddings[:1], 10)

#Bat dau do thoi gian chay
t0 = time.time()
for i in range(n_trials):
    _ = index.search(combined_embeddings[i:i + 1], 10)
elapsed_ms = (time.time() - t0) / n_trials * 1000

print(f"Thoi gian search trung binh: {elapsed_ms:.2f} ms/query tren {index.ntotal:,} vectors")

Thoi gian search trung binh: 14.17 ms/query tren 65,879 vectors
